In [1]:
import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

'0.6.6'

# Usage
- Configure the AWS 'default' profile for a user that has warehouse read/write permissions
- Adjust the `reference_time` timestamps in `filters` to your requested time window (current implementation supports quarters, e.g. Jan01-Mar31)

# Process
Starting with the joined timeseries table:
- Filter down to the configurations and time periods of interest
- Add row level calulated fields such as `forecast_leadtime_bin` and `quarter` (one new column), and time series aware calculated fields such as threshold exceedence (one column per threshold).  This requires the joined timeseries uniqueness fields be used. Total number of rows is unchanged.
- Stack (un-pivot) the data such that there is a threshold column that contains the threshold exceeded.  This increses the total number of rows of data, but allows our normal grouping/aggregates to be used.
- Aggregate based on the uniqueness fields plus row level calulated fields and threshold, to determine the min/median/max of each primary and secondary value within each `forecast_leadtime_bin` for each forecast.  This reduces the number of rows and adds a column per aggregation.
- Stack (un-pivot) the data again such that the min/mean/max columns are called primary_value and secondary_value and there is a `window_agg` column to identify which the value is.
- Lastly, we can group by location_id, configuration_name, etc. plus `forecast_leadtime_bin` and `window_agg` and calculate metrics with and without bootstrapping.

# Profiling
- All run with Bleeding Edge 4XL server.
- Desired bootstrap iterations: 1000
- Total locations: 7486

**2026-08-31: fixed the root cause of the crashes/nondeterminism** -- `unpack_results=True`
on the bootstrap metrics triggered a Spark `.first()` action per metric (9x) inside
`.aggregate()` itself, re-executing the whole upstream lazy DAG each time and racing
against spot-instance preemption -- exactly matching the `ShuffleMapStage ... (first at
.../teehr/querying/utils.py:207)` failures below. Replaced with a manual unpack after
aggregation (no Spark action). Also found `spark.sql.adaptive.coalescePartitions.enabled`
was collapsing the bootstrap pandas_udf stage down to ~2 tasks regardless of executor
count, since AQE coalesces based on shuffle byte size, not per-row compute cost -- disabled
for this workload. Rows below from before that point reflect the old, buggy pipeline.

| # Locations | # Days | # Bootstrap Iterations | Spark Params | Aggregation Time | Notes |
| --- | --- | --- | --- | --- | --- |
| All | 92 | 1000 | Cluster - 64 inst., 16g, 2 cores, 1024 part., 4g memOH, coalesce=false | 626s | %78 util. |
| 3000 | 92 | 1000 | Cluster - 32 inst., 12g, 2 cores, 512 part., no memOH, coalesce=false | 219s | 72% util. |
| 3000 | 92 | 1000 | Cluster - 32 inst., 12g, 2 cores, 256 part., no memOH, coalesce=false | 304s | 50% util. |
| 3000 | 92 | 1000 | Cluster - 96 inst., 12g, 2 cores, 1024 part., no memOH, coalesce=false | 230s | (see run_metrics above) |
| 3000 | 92 | 1000 | Cluster - 96 inst., 12g, 2 cores, 1024 part., no memOH, coalesce=false | 1194s | 75% util. |
| 3000 | 92 | 1000 | Cluster - 192 inst., 12g, 2 cores, 2048 part., no memOH, coalesce=false | 814s | 58% util. |
| 3000 | 92 | 1000 | Cluster - 192 inst., 12g, 2 cores, 1024 part., no memOH, coalesce=false | 826s | 57% util. |
| 3000 | 92 | 1000 | Cluster - 96 inst., 12g, 2 cores, 1024 part., no memOH, coalesce=false | 1241s | 76% util. |
| --- | --- | --- | --- | --- | --- |
| 3000 | 92 | 1000 | Cluster - 120 inst., 12g, 2 cores, 2560 part., no memOH, coalesce=false | 858s | (see run_metrics above) |
| 3000 | 92 | 1000 | Cluster - 240 inst., 12g, 2 cores, 2560 part., no memOH, coalesce=false | 1300s | (see run_metrics above) |
| 3000 | 92 | 1000 | Cluster - 240 inst., 12g, 2 cores, 2560 part., no memOH, coalesce=false | 710s | (see run_metrics above) |
| 1000 | 92 | 1000 | Cluster - 80 inst., 12g, 2 cores, 2560 part., no memOH, coalesce=false | 506s | (see run_metrics above)
| 100 | 92 | 1000 | Cluster - 8 inst., 12g, 2 cores, 256 part., no memOH, coalesce=false | 407s | (see run_metrics above) |
| 100 | 92 (2025-Q4) | 1000 | Cluster - 8 inst., 12g, 2 cores, 256 part., no memOH, coalesce=false | 323s | Success, first run after both fixes |
| All | 90 | 1000 | Cluster - 32 inst., 32g, 2 cores, 4096 part. | 52m | Success |
| All | 90 | 1000 | Cluster - 32 inst., 32g, 2 cores, 512 part., 8g mem. overhead | n/a | Began failing at stage 4. |
| All | 90 | 1000 | Cluster - 32 inst., 32g, 2 cores, 2048 part. | n/a | All good until failing at stage 191. |
| All | 90 | 10 | Cluster - 64 inst., 16g, 1 core, 2048 part. | n/a | Job aborted due to stage failure: ShuffleMapStage 30 (first at /srv/conda/envs/notebook/lib/python3.12/site-packages/teehr/querying/utils.py:207) has failed the maximum allowable number of times: 4. |
| All | 90 | 10 | Cluster - 32 inst., 32g, 1 core, 2048 part. | 1hr46m12s | |
| All | 365 | 100 | Cluster - 64 inst., 16g, 1 core, 2048 part. | n/a | OOM failures around stages 12-22 |
| All | 365 | 100 | Cluster - 32 inst., 32g, 2 cores, 2048 part. | n/a | OOM failures around stages 12-22 |
| All | 365 | 100 | Cluster - 32 inst., 32g, 2 cores, 2048 part. | n/a | Failures around stages 12-22 - executors deleted by a user or the framework. |
| All | 365 | 100 | Cluster - 128 inst., 32g, 2 cores, 1024 part. | n/a | Failures around stages 12-22 - executors deleted by a user or the framework. |
| All | 90 | 100 | Cluster - 128 inst., 32g, 2 cores, 4096 part. (no AEQ) | n/a | Failures around stages 12-22 - executors deleted by a user or the framework. |
| All | 90 | 100 | Cluster - 64 inst., 40g, 2 cores, 4096 part., 12g memOH (no AEQ) | n/a | Started seeing failures around stage ~50, cancelled to tweak |
| All | 90 | 100 | Cluster - 128 inst., 16g, 2 cores, 2048 part., 16g memOH | n/a | Acted funny around stage 18, cancelled to tweak |
| All | 90 (2025-Q3) | 100 | Cluster - 128 inst., 20g, 2 cores, 2048 part., 10g memOH | ~2.5 hrs (8907s) | Stg18: 12min |
| All | 270 | 100 | Cluster - 128 inst., 20g, 2 cores, 2048 part., 10g memOH |  | Stg18: ~20-30min - Failed around stage ~90 |
| All | 90 (2025-Q4) | 100 | Cluster - 128 inst., 20g, 2 cores, 2048 part., 10g memOH |  | Failed around stage ~90 |
| All | 90 (2025-Q4) | 100 | Cluster - 128 inst., 24g, 2 cores, 2048 part., 16g memOH |  | Failed around stage ~90 |
| All | 90 (2026-Q1) | 100 | Cluster - 128 inst., 24g, 2 cores, 2048 part., 16g memOH | ~2.5 hrs | Had to try write process multiple times |
| All | 90 (2026-Q2) | 100 | Cluster - 128 inst., 24g, 2 cores, 2048 part., 16g memOH | ~2.5 hrs | Had to try write process multiple times |
| All | 90 (2025-Q4) | 100 | Cluster - 128 inst., 24g, 2 cores, 2048 part., 16g memOH | ~2.5 hrs | Worked after previously failing with same config? |
| 3 | 30 | 10 | Cluster - 64 inst., 16g, 1 core, 2048 part. | 3m34s | |
| 3 | 90 | 10 | Cluster - 64 inst., 16g, 1 core, 2048 part. | 4m41s | |
| 3 | 30 | 100 | Cluster - 64 inst., 16g, 1 core, 2048 part. | 7m17s | |
| 3 | 90 | 100 | Cluster - 64 inst., 16g, 1 core, 2048 part. | 10m36s | |
| 3 | 30 | 10 | Default | 1m24s | |
| 3 | 90 | 10 | Default | 1m57s | |
| 3 | 30 | 100 | Default | 5m26s | |
| 3 | 90 | 100 | Default | 8m47s | |

In [2]:
import os

# Alternate executor pod template targeting the ON-DEMAND `nb-r5-4xlarge-teehr`
# node group instead of the spot `spark-r5-4xlarge-spot` pool, for tuning runs
# where we want clean measurements without spot-interruption noise. Same
# instance type (r5.4xlarge) so executor sizing math stays comparable to prior
# spot-based runs. Different taint on this node group (hub.jupyter.org/dedicated
# =user vs teehr-hub/dedicated=worker), so it needs its own tolerations.
ONDEMAND_POD_TEMPLATE_PATH = os.path.expanduser("~/executor-pod-template-ondemand.yaml")

with open(ONDEMAND_POD_TEMPLATE_PATH, "w") as f:
    f.write("""apiVersion: v1
kind: Pod
spec:
  terminationGracePeriodSeconds: 60
  securityContext:
    runAsUser: 1000
    runAsGroup: 1000
    fsGroup: 1000
  containers:
  - name: spark-kubernetes-executor
    securityContext:
      runAsUser: 1000
      runAsGroup: 1000
      allowPrivilegeEscalation: false
    lifecycle:
      preStop:
        exec:
          command: ["/bin/sh", "-c", "sleep 30"]
    volumeMounts:
    - name: data-nfs
      mountPath: /data
  volumes:
  - name: data-nfs
    persistentVolumeClaim:
      claimName: data-nfs
  tolerations:
  - effect: "NoSchedule"
    key: "hub.jupyter.org/dedicated"
    operator: "Equal"
    value: "user"
  - effect: "NoSchedule"
    key: "hub.jupyter.org_dedicated"
    operator: "Equal"
    value: "user"
  nodeSelector:
    teehr-hub/nodegroup-name: nb-r5-4xlarge
""")

print(f"Wrote alternate pod template to {ONDEMAND_POD_TEMPLATE_PATH}")


Wrote alternate pod template to /home/jovyan/executor-pod-template-ondemand.yaml


In [3]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=64,
    executor_memory="16g",
    executor_cores=2,
    aws_profile="default",
    pod_template_path=ONDEMAND_POD_TEMPLATE_PATH,
    update_configs={
        "spark.sql.shuffle.partitions": 1024,
        "spark.sql.adaptive.coalescePartitions.enabled": "false",
        "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
        "spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE": "vectorized",
        "spark.executor.memoryOverhead": "4g",
    }
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.3.91
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 64
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 16g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 2
INFO:teehr.evaluatio

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 37588)
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/srv/conda/envs/notebook/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/srv/conda/envs/notebook/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/srv/conda/envs/notebook/lib/python3.12/socketserver.py", line 766, in __init__
    self.handle()
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
                    

In [ ]:
spark.sql("DROP TABLE IF EXISTS nwmd_metrics_by_location_test PURGE")

In [4]:
# --- Resource-sizing instrumentation -----------------------------------------
# Pulls executor/stage summary metrics from the Spark REST API (no external deps,
# works as long as the Spark UI is enabled) so we can right-size the cluster from
# actual numbers instead of guessing. Call capture_spark_run_metrics(spark, ...)
# any time before spark.stop() -- it reads cumulative stats for the session so far.
import json
import urllib.request
from urllib.parse import urlparse
from datetime import datetime, timezone


def _spark_api_get(spark, path):
    ui = spark.sparkContext.uiWebUrl
    if not ui:
        raise RuntimeError("Spark UI is not enabled (uiWebUrl is None) -- can't fetch REST metrics.")
    app_id = spark.sparkContext.applicationId
    port = urlparse(ui).port or 4040
    # uiWebUrl often reports the driver's internal pod IP, which isn't always
    # reachable from within the driver's own process in this environment (seen:
    # "Connection refused"). Try it first, then fall back to loopback addresses
    # on the same port -- Spark's UI Jetty server binds to all interfaces.
    candidates = list(dict.fromkeys([ui, f"http://localhost:{port}", f"http://127.0.0.1:{port}"]))
    last_err = None
    for base in candidates:
        url = f"{base}/api/v1/applications/{app_id}{path}"
        try:
            with urllib.request.urlopen(url, timeout=10) as resp:
                return json.load(resp)
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Could not reach Spark UI REST API at any of {candidates}: {last_err}")


def spark_config_summary(spark):
    """Compact 'Spark Params' string matching the profiling table's column format."""
    return (
        f"{spark.conf.get('spark.executor.instances', '?')} inst., "
        f"{spark.conf.get('spark.executor.memory', '?')}, "
        f"{spark.conf.get('spark.executor.cores', '?')} cores, "
        f"{spark.conf.get('spark.sql.shuffle.partitions', '?')} part., "
        f"{spark.conf.get('spark.executor.memoryOverhead', 'no')} memOH, "
        f"coalesce={spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled', 'default')}"
    )


def _infer_days_from_filters(filters):
    """Best-effort day-count from reference_time filters, for the profiling row."""
    lo = hi = None
    for f in filters:
        if getattr(f, "column", None) != "reference_time":
            continue
        try:
            ts = datetime.fromisoformat(f.value)
        except Exception:
            continue
        if f.operator in (">", ">="):
            lo = ts if lo is None else min(lo, ts)
        elif f.operator in ("<", "<="):
            hi = ts if hi is None else max(hi, ts)
    return (hi - lo).days if lo and hi else "?"


def capture_spark_run_metrics(spark, label="run"):
    """Summarize executor/stage metrics for this Spark session so far.

    Surfaces exactly the signals needed to right-size a cluster: whether
    executors were lost mid-run (spot preemption), whether memory spilled to
    disk (undersized executor memory for the shuffle partition count), core
    counts (for utilization via report_utilization), and stage failure count.

    Note: peak/max memory here reflects Spark's on-heap *storage* memory pool
    (cache/broadcast), which is largely irrelevant for this notebook's
    pandas_udf-heavy bootstrap stage -- that stage's real memory pressure is
    off-heap Python worker memory, which isn't exposed by this REST endpoint.
    Treat memory-spill and failure/executor-loss counts as the trustworthy
    signals; treat the storage-memory-utilization note as informational only.
    """
    try:
        executors = _spark_api_get(spark, "/executors")
        stages_complete = _spark_api_get(spark, "/stages?status=complete")
        stages_failed = _spark_api_get(spark, "/stages?status=failed")
    except Exception as e:
        print(f"Could not fetch Spark REST metrics: {e}")
        return None

    # Exclude the driver entry -- it reports the driver pod's own (much larger,
    # unrelated) heap size, which otherwise skews max/peak memory calculations.
    worker_executors = [e for e in executors if e.get("id") != "driver"]
    active_executors = [e for e in worker_executors if e.get("isActive", True)]
    removed_executors = [e for e in worker_executors if not e.get("isActive", True)]

    total_gc_ms = sum(e.get("totalGCTime", 0) for e in worker_executors)
    total_duration_ms = sum(e.get("totalDuration", 0) for e in worker_executors)
    total_shuffle_read = sum(e.get("totalShuffleRead", 0) for e in worker_executors)
    total_shuffle_write = sum(e.get("totalShuffleWrite", 0) for e in worker_executors)
    peak_mem_used = max((e.get("memoryUsed", 0) for e in worker_executors), default=0)
    max_mem_avail = max((e.get("maxMemory", 0) for e in worker_executors), default=0)
    total_mem_spill = sum(s.get("memoryBytesSpilled", 0) for s in stages_complete)
    total_disk_spill = sum(s.get("diskBytesSpilled", 0) for s in stages_complete)
    total_cores = sum(e.get("totalCores", 0) for e in active_executors)

    summary = {
        "label": label,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "num_executors_seen": len(worker_executors),
        "num_executors_active": len(active_executors),
        "num_executors_removed": len(removed_executors),
        "total_cores_active": total_cores,
        "peak_executor_memory_used_gb": round(peak_mem_used / 1e9, 2),
        "executor_max_memory_gb": round(max_mem_avail / 1e9, 2),
        "total_gc_time_min": round(total_gc_ms / 1000 / 60, 2),
        "total_task_time_min": round(total_duration_ms / 1000 / 60, 2),
        "total_shuffle_read_gb": round(total_shuffle_read / 1e9, 2),
        "total_shuffle_write_gb": round(total_shuffle_write / 1e9, 2),
        "total_mem_spill_gb": round(total_mem_spill / 1e9, 2),
        "total_disk_spill_gb": round(total_disk_spill / 1e9, 2),
        "num_stages_completed": len(stages_complete),
        "num_stages_failed": len(stages_failed),
    }

    print(json.dumps(summary, indent=2))
    if removed_executors:
        print(
            f"WARNING: {len(removed_executors)} executor(s) were removed/lost during this "
            "run (spot preemption or similar) -- check the Spark UI Executors tab for cause."
        )
    if summary["num_stages_failed"] > 0:
        print(f"WARNING: {summary['num_stages_failed']} stage(s) failed during this run.")
    if summary["total_disk_spill_gb"] > 0:
        print(
            f"NOTE: {summary['total_disk_spill_gb']} GB spilled to disk -- executor memory "
            "may be undersized for the current shuffle partition count."
        )

    return summary


def report_utilization(run_metrics, wall_seconds):
    """Core utilization = total task-time / (wall_clock x active cores).

    Call after capture_spark_run_metrics with the same run's elapsed_seconds.
    Low utilization (well under 1.0) suggests too many cores/executors for the
    actual parallel work available (or a skew/coalescing bottleneck); high
    utilization near 1.0 means the cluster was busy essentially the whole time.
    """
    if not run_metrics or not run_metrics.get("total_cores_active"):
        print("No utilization data available.")
        return None
    core_minutes_available = wall_seconds / 60 * run_metrics["total_cores_active"]
    pct = run_metrics["total_task_time_min"] / core_minutes_available if core_minutes_available else 0
    print(f"Core utilization: {pct:.0%} ({run_metrics['total_task_time_min']:.1f} task-min / "
          f"{core_minutes_available:.1f} core-min available)")
    return pct

In [5]:
# --- Stage-attempt failure detail -----------------------------------------
# /stages?status=failed only reports stages whose FINAL status is failed -- a
# stage that fails once and succeeds on retry shows as "complete" overall, so
# it's invisible there even though the retry cost real wall-clock time. This
# queries each stage's full attempt history (including successful-after-retry
# ones) via /stages/{stageId} and surfaces the actual failureReason per failed
# attempt, so we don't need to read it off the Spark UI by hand.
def get_stage_attempt_failures(spark, max_stages=200):
    stage_summaries = (
        _spark_api_get(spark, "/stages?status=complete")
        + _spark_api_get(spark, "/stages?status=failed")
    )
    stage_ids = sorted({s["stageId"] for s in stage_summaries})[:max_stages]

    failures = []
    for stage_id in stage_ids:
        try:
            attempts = _spark_api_get(spark, f"/stages/{stage_id}")
        except Exception as e:
            print(f"Could not fetch stage {stage_id}: {e}")
            continue
        if not isinstance(attempts, list):
            attempts = [attempts]
        for a in attempts:
            if a.get("status") == "FAILED" or a.get("failureReason"):
                failures.append({
                    "stageId": stage_id,
                    "attemptId": a.get("attemptId"),
                    "status": a.get("status"),
                    "numCompleteTasks": a.get("numCompleteTasks"),
                    "numFailedTasks": a.get("numFailedTasks"),
                    "failureReason": a.get("failureReason"),
                })

    if not failures:
        print("No failed stage attempts found (checked stage IDs: "
              f"{stage_ids[0]}-{stage_ids[-1]}).")
        return []

    print(f"Found {len(failures)} failed stage attempt(s):\n")
    for f in failures:
        print(f"Stage {f['stageId']} attempt {f['attemptId']} "
              f"({f['numCompleteTasks']} complete / {f['numFailedTasks']} failed tasks):")
        print(f"  {f['failureReason']}\n")
    return failures

In [6]:
start = time.perf_counter()

In [7]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


In [8]:
joined_cols = ev.table("fcst_joined_timeseries").to_sdf().columns
non_unique_fields = ['primary_value','secondary_value','created_at','updated_at', "value_time"]
uniquenes_fields = [c for c in joined_cols if c not in non_unique_fields]
# uniquenes_fields

INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.


In [9]:
# ids = ev.locations.filter("id like 'usgs-%'").to_sdf().select("id")
# sample = ids.sample(False, 0.5, seed=456).limit(3000).collect()
# location_ids = [r.id for r in ids.collect()]
# print(len(location_ids))

# spark.sql("""
# USE iceberg.teehr
# """)
# rows = spark.sql("""
# SELECT distinct primary_location_id FROM fcst_joined_timeseries
# """).collect()
# location_ids = [r.primary_location_id for r in rows]
# print(len(location_ids))

In [10]:
filters = [
    TableFilter(
        column="configuration_name",
        operator="=",
        value="nwm30_medium_range"
    ),
    TableFilter(
        column="reference_time",
        operator=">=",
        value="2025-10-01T00:00",
    ),
    TableFilter(
        column="reference_time",
        operator="<",
        value="2026-10-01T00:00",
    ),
    # TableFilter(
    #     column="primary_location_id",
    #     operator="in",
    #     value=location_ids
    # )
]

In [ ]:
# Define the above percentile event detection calculated fields for 85th, 95th, and 99th percentiles.
# Note: both the threshold and event detection are based on the primary_value field.  
# This may differ from the way it is done in the NWM Explorer.  Does the NWM Explorer use the primary_value 
# of the threshold definition but the secondary_value field for event detection?

remove_for_quantiles = ["secondary_location_id", "reference_time", "member"]
quantile_group = [c for c in uniquenes_fields if c not in remove_for_quantiles]

calculated_fields = [
    rcf.GenericSQL(
        output_field_name="quarter",
        sql_statement="CONCAT(YEAR(reference_time), '-Q', QUARTER(reference_time))"
    ),
    rcf.ForecastLeadTimeBins(
        bin_size=pd.Timedelta(hours=24),
        output_field_name="forecast_lead_time_bin"
    ),
    tcf.AbovePercentileEventDetection(
        quantile=0.85,
        output_event_field_name="above_q85",
        skip_event_id=True,
        value_field_name="primary_value",
        uniqueness_fields=quantile_group
    ),
    tcf.AbovePercentileEventDetection(
        quantile=0.95,
        output_event_field_name="above_q95",
        skip_event_id=True,
        value_field_name="primary_value",
        uniqueness_fields=quantile_group
    ),
    tcf.AbovePercentileEventDetection(
        quantile=0.99,
        output_event_field_name="above_q99",
        skip_event_id=True,
        value_field_name="primary_value",
        uniqueness_fields=quantile_group
    )
]

In [12]:
# Get raw joined timeseries
tbl = ev.table("fcst_joined_timeseries").filter(filters).add_calculated_fields(calculated_fields)

INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter [TableFilter(column='configuration_name', operator=<FilterOperators.eq: '='>, value='nwm30_medium_range'), TableFilter(column='reference_time', operator=<FilterOperators.gte: '>='>, value='2025-10-01T00:00'), TableFilter(column='reference_time', operator=<FilterOperators.lt: '<'>, value='2026-10-01T00:00')].


In [13]:
# len(tbl.distinct_values("primary_location_id"))

In [14]:
# Stack thresholds
threshold_cols = ["above_q85", "above_q95", "above_q99"]
threshold_stack_base_cols = [c for c in tbl.columns if c not in threshold_cols]
# threshold_stack_base_cols

In [15]:
joined_timeseries_with_thresholds_tbl = (
    tbl.selectExpr(
        *threshold_stack_base_cols,
        """
        stack(
            4,
            cast(null as string), true,
            'above_q85', above_q85,
            'above_q95', above_q95,
            'above_q99', above_q99
        ) as (threshold, keep_row)
        """
    )
    .where("keep_row")
    .select(*threshold_stack_base_cols, "threshold")   # no .drop()
)

# print(f"no threshold_rows: {joined_timeseries_with_thresholds_tbl.where("threshold is NULL").count()}")
# print(f"threshold_rows: {joined_timeseries_with_thresholds_tbl.where("threshold is not NULL").count()}")
# print(f"total: {joined_timeseries_with_thresholds_tbl.count()}")


In [16]:
# Add window aggregations
window_metrics = [
    s.Average(
        primary_field_name="primary_value",
        output_field_name="mean_primary_value"
    ),
    s.Average(
        primary_field_name="secondary_value",
        output_field_name="mean_secondary_value"
    ),
    s.Minimum(
        primary_field_name="primary_value",
        output_field_name="min_primary_value"
    ),
    s.Minimum(
        primary_field_name="secondary_value",
        output_field_name="min_secondary_value"
    ),
    s.Maximum(
        primary_field_name="primary_value",
        output_field_name="max_primary_value"
    ),
    s.Maximum(
        primary_field_name="secondary_value",
        output_field_name="max_secondary_value"
    ),
    s.Count(
        primary_field_name="secondary_value",
        output_field_name="n_in_bin"
    )
]

In [17]:
group_by_bin = [*uniquenes_fields, "quarter", "forecast_lead_time_bin", "threshold"]
group_by_bin

['reference_time',
 'primary_location_id',
 'secondary_location_id',
 'configuration_name',
 'unit_name',
 'variable_name',
 'member',
 'quarter',
 'forecast_lead_time_bin',
 'threshold']

In [18]:
bin_aggs_tbl = joined_timeseries_with_thresholds_tbl.aggregate(
    group_by=group_by_bin,
    metrics=window_metrics
)
# print(f"bin_aggs: {bin_aggs_tbl.count()}")

INFO:teehr.evaluation.dataframe_base:Performing the aggregation.


In [19]:
# after your bin aggregation
# bin_aggs_tbl.select("n_in_bin").summary().show()

In [20]:
pivoted_bin_aggs_tbl = bin_aggs_tbl.selectExpr(
    *group_by_bin,
    """
    stack(
        3,
        'mean', mean_primary_value, mean_secondary_value,
        'min',  min_primary_value,  min_secondary_value,
        'max',  max_primary_value,  max_secondary_value
    ) as (window_agg, primary_value, secondary_value)
    """
)
# print(f"pivoted_bin_aggs: {pivoted_bin_aggs_tbl.count()}")
# pivoted_bin_aggs_tbl.columns

In [21]:
# pivoted_bin_aggs_tbl.show()

In [22]:
# pivoted_bin_aggs_tbl.distinct_values("primary_location_id")
# pivoted_bin_aggs_tbl.distinct_values("reference_time")

In [23]:
# Nash-Sutcliffe efficiency
# Relative mean bias
# Pearson correlation coefficient
# Kling-Gupta efficiency

# Relative mean
# Relative median
# Relative minimum
# Relative maximum
# Relative standard deviation


# Configure bootstrap
bootstap = bs.Stationary(
    reps=1000,
    # block_size=100,
    seed=1234,
    quantiles=[0.025, 0.975]
)

In [24]:
metrics = [
    s.Count(),
    s.Average(),
    s.Minimum(),
    s.Maximum(),
    dm.RelativeMean(),
    dm.RelativeMedian(),
    dm.RelativeMinimum(),
    dm.RelativeMaximum(),
    dm.RelativeStandardDeviation(),
    dm.RelativeBias(
        add_epsilon=True,
    ),
    dm.NashSutcliffeEfficiency(
        add_epsilon=True,
    ),
    dm.KlingGuptaEfficiency(
        add_epsilon=True,
    ),
    dm.PearsonCorrelation(
        add_epsilon=True,
    ),
    # NOTE: unpack_results is intentionally NOT set on the bootstrap metrics below.
    # teehr's default unpack path (post_process_metric_results -> unpack_sdf_dict_columns)
    # calls sdf.select(column_name).first() once per metric with unpack_results=True -- a
    # real Spark action that retriggers the entire upstream lazy DAG once per metric (9x
    # here) and is the confirmed cause of the "ShuffleMapStage ... first at
    # teehr/querying/utils.py:207" crashes and nondeterministic same-config failures seen
    # in the profiling table above. We unpack manually after aggregation instead (see the
    # unpack_quantile_bootstrap_columns cell below), which needs no Spark action since the
    # quantile keys are already known statically from `bootstap.quantiles`.
    dm.RelativeMean(
        output_field_name="relative_mean_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeMedian(
        output_field_name="relative_median_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeMinimum(
        output_field_name="relative_minimum_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeMaximum(
        output_field_name="relative_maximum_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeStandardDeviation(
        output_field_name="relative_standard_deviation_boot",
        bootstrap=bootstap,
    ),
    dm.NashSutcliffeEfficiency(
        output_field_name="nash_sutcliffe_efficiency_boot",
        bootstrap=bootstap,
    ),
    dm.RelativeBias(
        output_field_name="relative_bias_boot",
        bootstrap=bootstap,
    ),
    dm.PearsonCorrelation(
        output_field_name="pearson_correlation_boot",
        bootstrap=bootstap,
    ),
    dm.KlingGuptaEfficiency(
        output_field_name="kling_gupta_efficiency_boot",
        bootstrap=bootstap,
    ),
]

In [25]:
group_by = [
    "primary_location_id",
    "secondary_location_id",
    "configuration_name",
    "unit_name",
    "variable_name",
    "member",
    "quarter",
    "forecast_lead_time_bin",
    "threshold",
    "window_agg",
]

In [26]:
%%time
results = pivoted_bin_aggs_tbl.aggregate(
    group_by=group_by,
    metrics=metrics
)

INFO:teehr.evaluation.dataframe_base:Performing the aggregation.


CPU times: user 197 ms, sys: 71 ms, total: 268 ms
Wall time: 1.24 s


In [27]:
# Manually unpack the bootstrap quantile MapType columns instead of relying on
# `unpack_results=True` (see note above the `metrics` list for why: the default
# unpack path triggers a Spark .first() action per metric that retriggers the
# whole upstream DAG). Quantile keys are known statically from `bootstap.quantiles`,
# so no action is needed here -- this stays fully lazy.
#
# NOTE: `results` is a teehr BaseTable, not a plain PySpark DataFrame. BaseTable.drop()
# is a different method (drops the underlying warehouse table, no column arg) that
# shadows PySpark's DataFrame.drop(*cols) even with enable_spark_proxy=True, since the
# proxy in __getattr__ only kicks in when the attribute isn't already defined on the
# class. So we do the column manipulation on the raw sdf via .to_sdf() and rewrap once
# with ._with_sdf() at the end, which is the same internal pattern order_by/aggregate/
# add_geometry already use.
def unpack_quantile_bootstrap_columns(table, metrics):
    sdf = table.to_sdf()
    for m in metrics:
        if not getattr(m, "bootstrap", None):
            continue
        for q in m.bootstrap.quantiles:
            key = f"{m.output_field_name}_{q}"
            sdf = sdf.withColumn(key.replace(".", "_"), F.col(m.output_field_name).getItem(key))
        sdf = sdf.drop(m.output_field_name)
    return table._with_sdf(sdf)

results = unpack_quantile_bootstrap_columns(results, metrics)

In [28]:
results = results.order_by(group_by).add_geometry()

INFO:teehr.evaluation.dataframe_base:Setting order_by ['primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member', 'quarter', 'forecast_lead_time_bin', 'threshold', 'window_agg'].
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.


In [29]:
%%time
results.explain(mode="simple")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [primary_location_id#601, secondary_location_id#602, configuration_name#603, unit_name#604, variable_name#605, member#606, quarter#607, forecast_lead_time_bin#608, threshold#609, window_agg#610, count#123L, average#124, minimum#125, maximum#126, relative_mean#341, relative_median#342, relative_minimum#343, relative_maximum#344, relative_standard_deviation#345, relative_bias#346, nash_sutcliffe_efficiency#347, kling_gupta_efficiency#348, pearson_correlation#349, relative_mean_boot_0_025#611, relative_mean_boot_0_975#612, ... 18 more fields]
   +- BroadcastHashJoin [primary_location_id#601], [primary_location_id#640], Inner, BuildRight, false
      :- Project [coalesce(primary_location_id#506, primary_location_id#572) AS primary_location_id#601, coalesce(secondary_location_id#507, secondary_location_id#544) AS secondary_location_id#602, coalesce(configuration_name#508, configuration_name#573) AS configuration_name#603, co

In [30]:
# %%time
# results.show()

In [31]:
# NOTE: pointed at a *_test table while validating the unpack_results/.first() fix so we
# don't overwrite the useful existing results in nwmd_metrics_by_location. Repoint back to
# "nwmd_metrics_by_location" only after full-scale validation succeeds consistently.
table_name = "nwmd_metrics_by_location_test"

nullables = ["member", "threshold"]
table_exists = ev.spark.catalog.tableExists(f"iceberg.teehr.{table_name}")

if table_exists:
    results.write_to(
        table_name=table_name,
        write_mode="upsert",
        uniqueness_fields=[column for column in group_by if column not in nullables],
        nullable_fields=nullables,
        partition_by=["quarter"],
    )
else:
    results.write_to(
        table_name=table_name,
        write_mode="create_or_replace",
        partition_by=["quarter"],
    )

metric_columns = [metric.output_field_name for metric in metrics]

properties = {
    "description": "NWM diagnostics metrics by location ID",
    "group_by": ", ".join(group_by),
    "metrics": ", ".join(metric_columns)
}

for key, value in properties.items():
    ev.spark.sql(f"""
        ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
    """)


INFO:teehr.evaluation.dataframe_base:Writing to table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location_test.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.nwmd_metrics_by_location_test.
INFO:teehr.evaluation.write:Start writing to warehouse table 'nwmd_metrics_by_location_test'.
INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location_test.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.nwmd_metrics_by_location_test.


SparkRuntimeException: [MERGE_CARDINALITY_VIOLATION] The ON search condition of the MERGE statement matched a single row from the target table with multiple rows of the source table.
This could result in the target row being operated on more than once with an update or delete operation and is not allowed. SQLSTATE: 23K01

In [ ]:
end = time.perf_counter()

elapsed_seconds = end - start
print(f"{elapsed_seconds:.6f} s")

In [ ]:
# Capture resource-usage metrics for this run BEFORE spark.stop() (the REST API
# stops responding once the session ends). Paste the printed markdown row into
# the Profiling table above to keep a running record.
n_locations = len(location_ids) if "location_ids" in globals() else "All"
n_days = _infer_days_from_filters(filters)

run_metrics = capture_spark_run_metrics(spark, label=f"{n_locations} locations, {n_days} days")
report_utilization(run_metrics, elapsed_seconds)

print(
    f"\n| {n_locations} | {n_days} | {bootstap.reps} | Cluster - {spark_config_summary(spark)} "
    f"| {elapsed_seconds:.0f}s | (see run_metrics above) |"
)

In [ ]:
# Run this against the still-live session (spark.stop() is commented out below)
# to see the actual failure reason for the retried/failed stages from this run,
# without needing to read it off the Spark UI by hand.
stage_failures = get_stage_attempt_failures(spark)

In [1]:
spark.stop()

NameError: name 'spark' is not defined